# Build AIMS4PT Cpx Temperature Workflow

This notebook builds the temperature-model workflow, including model-pool setup, OOD diagnostics, temperature-deviation prediction


# Setup and input data

In [ ]:
from pathlib import Path
import sys

def _find_project_root(start=None):
    """Find the project root by walking upward from the current directory."""
    path = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml above the current directory.")

PROJECT_ROOT = _find_project_root()
PAPER_DIR = PROJECT_ROOT / "paper"
CACHE_DIR = PAPER_DIR / ".cache"
IMAGES_DIR = PAPER_DIR / ".images"
DATA_DIR = PAPER_DIR / "data"
for _dir in (CACHE_DIR, IMAGES_DIR, DATA_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
import os
import time
import importlib
import matplotlib.pyplot as plt
from aims4pt.model_tools.model_registry import  get_models_initial_pools, print_all_registered_thermobarometry, ALL_MODELS_MODULES
from aims4pt.toolkit_utils import get_file_path
for module in ALL_MODELS_MODULES:
    importlib.import_module(module)




calculate_shap = False # make sure this to False when not needed, as it takes time and memory

timestamp = time.strftime("%Y%m%d_%H%M")

In [3]:
unseen_experiments_path = r"C:\Users\13493\Documents\Thermobarometers\Database\data\cross_check_data\cpx_liq_unseen_unique_testv2__2025-1-8.xlsx"
unseen_experiments_path = r"C:\Users\13493\Documents\Thermobarometers\Database\data\cross_check_data\unique_cpx_liq_auto_filtered.xlsx"# This version is not filtered by the 10 kbar threshold.
#change to your path

unseen_experiments_df = pd.read_excel(unseen_experiments_path)

#col names for unseen_experiments_df
cpx_names = ['SiO2_cpx', 'TiO2_cpx', 'Al2O3_cpx',
       'Fe2O3_cpx', 'Cr2O3_cpx', 'FeO_cpx', 'MnO_cpx', 'MgO_cpx', 'NiO_cpx',
       'CoO_cpx', 'CaO_cpx', 'Na2O_cpx', 'K2O_cpx', 'P2O5_cpx',]
liq_names = ['SiO2_liq', 'TiO2_liq', 'Al2O3_liq',
       'Fe2O3_liq', 'Cr2O3_liq', 'FeO_liq', 'MnO_liq', 'MgO_liq', 'NiO_liq',
       'CoO_liq', 'CaO_liq', 'Na2O_liq', 'K2O_liq', 'P2O5_liq']
P_col = 'P (kbar)'
T_col = 'T (C)'


## clean


In [4]:
print("Before cleaning, unseen_experiments_df shape:", unseen_experiments_df.shape)
cpx_total = unseen_experiments_df[cpx_names].sum(axis=1)
unseen_experiments_df = unseen_experiments_df[(cpx_total>98)&(cpx_total<102)].reset_index(drop=True)
print("After total, unseen_experiments_df shape:", unseen_experiments_df.shape)
from aims4pt.data_tools.compositions import cpx_calculation
cpx_params = cpx_calculation(unseen_experiments_df[cpx_names])
MdivT= cpx_params["(Ca+Fe+Mg)/Si"]
unseen_experiments_df = unseen_experiments_df[(MdivT>0.9)&(MdivT<1.1)].reset_index(drop=True)
print("After cpx MdivT, unseen_experiments_df shape:", unseen_experiments_df.shape)
# kd filter
from aims4pt.data_tools.equilibrium import kdEquilibrium_test
mask, kd_values = kdEquilibrium_test(
    unseen_experiments_df[cpx_names],
    unseen_experiments_df[liq_names],
    kd = 0.28,
    error = 0.08,
    mode='Fe-Mg',)
unseen_experiments_df = unseen_experiments_df[mask].reset_index(drop=True)
print("After kd filter, unseen_experiments_df shape:", unseen_experiments_df.shape)
liq_total = unseen_experiments_df[liq_names].sum(axis=1)
liq_total.describe()

Before cleaning, unseen_experiments_df shape: (517, 36)
After total, unseen_experiments_df shape: (497, 36)
After cpx MdivT, unseen_experiments_df shape: (441, 36)
After kd filter, unseen_experiments_df shape: (293, 36)


count    293.000000
mean      98.324335
std        2.691512
min       86.971390
25%       97.802000
50%       99.813648
75%      100.000000
max      100.660000
dtype: float64

In [5]:

cpx_unseen = unseen_experiments_df[cpx_names].copy()
liq_unseen = unseen_experiments_df[liq_names].copy()
meta_unseen = unseen_experiments_df[[P_col, T_col]].copy()
# stratified split
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
P_values = unseen_experiments_df[P_col]
target_bin_size = 50
num_bins = max(2, min(10, len(P_values) // target_bin_size))
P_binned = pd.qcut(P_values, q=num_bins, labels=False, duplicates="drop")
(strat_train_idx, strat_test_idx), = sss.split(unseen_experiments_df, P_binned)
training_unseen_id = unseen_experiments_df.iloc[strat_train_idx].index
testing_unseen_id = unseen_experiments_df.iloc[strat_test_idx].index

#random split
# from sklearn.model_selection import train_test_split
# training_unseen_id, testing_unseen_id = train_test_split(
#     unseen_experiments_df.index,
#     test_size=0.2,
#     random_state=42,
#     )


# Initial all thermobarometry models

In [15]:
P_model_pool = get_models_initial_pools("P", "both" , False) # get all P models including cpx_only and cpx_liq
P_model_names = list(model.model_name for model in P_model_pool)
T_model_pool = get_models_initial_pools("T", "both" , False) # get all T models including cpx_only and cpx_liq
T_model_names = list(model.model_name for model in T_model_pool)


c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Putirka, 2008 eq32d_T; eq32a_P (cpx_only)_P_shap_df_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Putirka, 2008 eq32d_T; eq32a_P (cpx_only)_P_feature_importance_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\P_Putirka, 2008 eq32d_T; eq32a_P (cpx_only)_deviation_function_v_ind.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\OOD_detectors\Putirka, 2008 eq32d_T; eq32a_P (cpx_only)_P_weighted.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Putirka, 2008 eq32d_T; eq32b_P (cpx_only)_P_shap_df_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Putirka, 2008 eq32d_T; eq32b_P (cpx_only)_P_feature_importance_default.pkl
c:\users\13493\d

c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_P_cpx_train.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_P_liq_train.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_P_cpx_test.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_P_liq_test.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Ágreda-López et al., 2024 (cpx_only)_P_shap_df_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Ágreda-López et al., 2024 (cpx_only)_P_feature_importance_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\P_Ágreda-López et al., 2024 (cpx_only)_deviation_function_v_ind.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis

c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_P_cpx_train.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_P_liq_train.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_P_cpx_test.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_P_liq_test.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Ágreda-López et al., 2024 (cpx_liq)_P_shap_df_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Ágreda-López et al., 2024 (cpx_liq)_P_feature_importance_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\P_Ágreda-López et al., 2024 (cpx_liq)_deviation_function_v_ind.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_to

c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_T_cpx_train.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_T_liq_train.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_T_cpx_test.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_T_liq_test.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Ágreda-López et al., 2024 (cpx_only)_T_shap_df_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Ágreda-López et al., 2024 (cpx_only)_T_feature_importance_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Ágreda-López et al., 2024 (cpx_only)_deviation_function_v_ind.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis

c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_T_cpx_train.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_T_liq_train.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_T_cpx_test.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\data\Agreda2024\datapkl\X_T_liq_test.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Ágreda-López et al., 2024 (cpx_liq)_T_shap_df_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Ágreda-López et al., 2024 (cpx_liq)_T_feature_importance_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Ágreda-López et al., 2024 (cpx_liq)_deviation_function_v_ind.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_to

In [16]:
P_model_dict = {model.model_name: model for model in P_model_pool}
T_model_dict = {model.model_name: model for model in T_model_pool}

## Temperature SHAP feature importance

### Temperature model

In [26]:
first_time_run_shap = True
T_SHAP_df_dict = {}
T_feature_importance_dict = {}
import pickle
import aims4pt.model_tools.trained_model.SHAP as SHAP_package

# stratified sampling based on P distribution
from sklearn.model_selection import StratifiedShuffleSplit

if first_time_run_shap:
    for model, model_name in zip(T_model_pool, T_model_names):
        print(f"Processing model: {model_name}")
        if model.X_cpx_training is None:
            continue
        path = get_file_path(SHAP_package, f"{model_name}_T_shap_df_default.pkl")
        path_fi = get_file_path(SHAP_package, f"{model_name}_T_feature_importance_default.pkl")
        if os.path.exists(path) and os.path.exists(path_fi):
            with open(path, "rb") as f:
                shap_df = pickle.load(f)
            with open(path_fi, "rb") as f:
                feature_importance_df = pickle.load(f)
            T_SHAP_df_dict[model_name] = shap_df
            T_feature_importance_dict[model_name] = feature_importance_df
            continue

        # sampling 20% of training data for SHAP calculation
        len_training = model.X_cpx_training.shape[0]
        strat_split = StratifiedShuffleSplit(
            n_splits=1, test_size=0.2, random_state=42
        )

        # create bins for P values
        T_values = model.X_cpx_training["T_C"]
        num_bins = max(2, min(10, int(len_training / 50)))
        T_binned = pd.cut(T_values, bins=num_bins, labels=False)
        (strat_train_idx, strat_test_idx), = strat_split.split(model.X_cpx_training, T_binned)
        sampled_indices = strat_test_idx
        
        X_cpx_sampled = model.X_cpx_training.iloc[sampled_indices]
        X_liq_sampled = model.X_liq_training.iloc[sampled_indices]

        shap_df, feature_importance_df, shap_values =model.shap_calculation(
            X_cpx_sampled, X_liq_sampled, background_data=model.X_cpx_training, bg_liq=model.X_liq_training,
                                    package_predict_func=True, sampling_bg=100)
        T_SHAP_df_dict[model_name] = shap_df
        T_feature_importance_dict[model_name] = feature_importance_df
        with open(path, "wb") as f:
            pickle.dump(shap_df, f)
        with open(path_fi, "wb") as f:
            pickle.dump(feature_importance_df, f)
        # model.set_key_features("automated_accumulate")
else:
    for model_name in T_model_names:
        try:
            path = get_file_path(SHAP_package, f"{model_name}_T_shap_df_default.pkl")
            path_fi = get_file_path(SHAP_package, f"{model_name}_T_feature_importance_default.pkl")
            with open(path, "rb") as f:
                shap_df = pickle.load(f)
            with open(path_fi, "rb") as f:
                feature_importance_df = pickle.load(f)
        except FileNotFoundError:
            continue
        T_SHAP_df_dict[model_name] = shap_df
        T_feature_importance_dict[model_name] = feature_importance_df


        

Processing model: Putirka, 2008 eq32d_T; eq32a_P (cpx_only)
Processing model: Putirka, 2008 eq32d_T; eq32b_P (cpx_only)
Processing model: Petrelli et al., 2020 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Petrelli et al., 2020 (cpx_only)_T_shap_df_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Petrelli et al., 2020 (cpx_only)_T_feature_importance_default.pkl
Processing model: Higgins et al., 2021 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Higgins et al., 2021 (cpx_only)_T_shap_df_default.pkl
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Higgins et al., 2021 (cpx_only)_T_feature_importance_default.pkl
Processing model: Jorgenson et al., 2022 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Jorgenson et al., 2022 (cpx

## Temperature weighted OOD detector

### Temperature model

In [29]:
from aims4pt.statistic_tools.out_of_distribution import ThermobarometerOODFactory
import pickle
first_time_run_OOD_key_detector = True
import aims4pt.model_tools.trained_model.OOD_detectors as OOD_detectors_package
OOD_detectors_T_weighted_dict = {}
if first_time_run_OOD_key_detector:
    for model, model_name in zip(T_model_pool, T_model_names):
        print(f"Processing model: {model_name}")
        if model.X_cpx_training is None:
            continue
        path = get_file_path(OOD_detectors_package, f"{model_name}_T_weighted.pkl")
        # if os.path.exists(path):
        #     with open(path, "rb") as f:
        #         detector = pickle.load(f)
        #     OOD_detectors_T_weighted_dict[model_name] = detector
        #     continue
        model.feature_importance_df = T_feature_importance_dict[model_name]
        model.set_key_features(mode="all")
        detector = ThermobarometerOODFactory(model, "key", weighted=True)
        model.OOD_detector = detector
        OOD_detectors_T_weighted_dict[model_name] = detector
        with open(path, "wb") as f:
            pickle.dump(detector, f)
else:
    for model_name in P_model_names:
        try:
            path = get_file_path(OOD_detectors_package, f"{model_name}_T_weighted.pkl")
            with open(path, "rb") as f:
                detector = pickle.load(f)
        except FileNotFoundError:
            continue
        OOD_detectors_T_weighted_dict[model_name] = detector
        

   

Processing model: Putirka, 2008 eq32d_T; eq32a_P (cpx_only)
Processing model: Putirka, 2008 eq32d_T; eq32b_P (cpx_only)
Processing model: Petrelli et al., 2020 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\OOD_detectors\Petrelli et al., 2020 (cpx_only)_T_weighted.pkl
Processing model: Higgins et al., 2021 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\OOD_detectors\Higgins et al., 2021 (cpx_only)_T_weighted.pkl
Processing model: Jorgenson et al., 2022 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\OOD_detectors\Jorgenson et al., 2022 (cpx_only)_T_weighted.pkl
Processing model: Ágreda-López et al., 2024 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\OOD_detectors\Ágreda-López et al., 2024 (cpx_only)_T_weighted.pkl
Processing model: Wang et al., 2021 (cpx_only)
c:\users\13493\documents\

## Temperature deviation predictor

In [ ]:
first_time_run_deviation_fit = True


from aims4pt.model_tools.deviation_models import deviation_model_for_a_thermobarometer
import aims4pt.model_tools.trained_model.Deviation_functions as Deviation_functions_package
r2_collect_P = {}
r2_collect_T = {}
deviation_functions_P_dict = {}
deviation_functions_T_dict = {}
# indices = [4, -3, -2] # ((P_model_pool[i], P_model_names[i]) for i in indices)
calib_cpx = cpx_unseen.loc[training_unseen_id]
calib_P = meta_unseen.loc[training_unseen_id, P_col]
calib_T = meta_unseen.loc[training_unseen_id, T_col]
calib_liq = liq_unseen.loc[training_unseen_id]
if first_time_run_deviation_fit:
    for model, model_name in zip(P_model_pool, P_model_names):
        print(f"Processing deviation function for model: {model_name}")
        save_path = get_file_path(Deviation_functions_package, f"P_{model_name}_deviation_function_v_ind.pkl")
        # if os.path.exists(save_path):
        #     with open(save_path, "rb") as f:
        #         deviation_function = pickle.load(f)
        #     deviation_functions_P_dict[model_name] = deviation_function
        #     r2_collect_P[model_name] = deviation_function.r2
        #     continue
        deviation_function = deviation_model_for_a_thermobarometer(model)
        try:
            ood_detector = OOD_detectors_P_weighted_dict[model_name]
            ood_mask = ood_detector.is_ood(calib_cpx,calib_liq)
        except KeyError:
            ood_mask = np.zeros(len(calib_cpx), dtype=bool)
            print(f"No OOD detector found for model: {model_name}, proceeding without OOD filtering.")
        deviation_function.fit_deviation_model(calib_cpx[~ood_mask],
                                                calib_P[~ood_mask],
                                                calib_liq[~ood_mask])
                                
        deviation_functions_P_dict[model_name] = deviation_function
        model.deviation_function = deviation_function
        r2_collect_P[model_name] = deviation_function.r2
        with open(save_path, "wb") as f:
            pickle.dump(deviation_function, f)
    
    
    for model, model_name in zip(T_model_pool, T_model_names):
        print(f"Processing deviation function for model: {model_name}")
        save_path = get_file_path(Deviation_functions_package, f"T_{model_name}_deviation_function_v_ind.pkl")

        deviation_function = deviation_model_for_a_thermobarometer(model)
        try:
            ood_detector = OOD_detectors_T_weighted_dict[model_name]
            ood_mask = ood_detector.is_ood(calib_cpx,calib_liq)
        except KeyError:
            ood_mask = np.zeros(len(calib_cpx), dtype=bool)
            print(f"No OOD detector found for model: {model_name}, proceeding without OOD filtering.")
        deviation_function.fit_deviation_model(calib_cpx[~ood_mask],
                                                calib_T[~ood_mask],
                                                calib_liq[~ood_mask])
        deviation_functions_T_dict[model_name] = deviation_function
        model.deviation_function = deviation_function
        r2_collect_T[model_name] = deviation_function.r2
        with open(save_path, "wb") as f:
            pickle.dump(deviation_function, f)

else:

    for model_name in P_model_names:
        load_path = get_file_path(Deviation_functions_package, f"P_{model_name}_deviation_function_v_ind.pkl")
        with open(load_path, "rb") as f:
            deviation_function = pickle.load(f)
        deviation_functions_P_dict[model_name] = deviation_function
        r2_collect_P[model_name] = deviation_function.r2
    

    for model_name in T_model_names:
        load_path = get_file_path(Deviation_functions_package, f"T_{model_name}_deviation_function_v_ind.pkl")
        with open(load_path, "rb") as f:
            deviation_function = pickle.load(f)
        deviation_functions_T_dict[model_name] = deviation_function
        r2_collect_T[model_name] = deviation_function.r2

# print(pd.DataFrame.from_dict(r2_collect_P, orient='index', columns=['R²']))
# print(pd.DataFrame.from_dict(r2_collect_T, orient='index', columns=['R²']))

Processing deviation function for model: Putirka, 2008 eq32d_T; eq32a_P (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Putirka, 2008 eq32d_T; eq32a_P (cpx_only)_deviation_function_v_ind.pkl
No OOD detector found for model: Putirka, 2008 eq32d_T; eq32a_P (cpx_only), proceeding without OOD filtering.


c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:

Processing deviation function for model: Putirka, 2008 eq32d_T; eq32b_P (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Putirka, 2008 eq32d_T; eq32b_P (cpx_only)_deviation_function_v_ind.pkl
No OOD detector found for model: Putirka, 2008 eq32d_T; eq32b_P (cpx_only), proceeding without OOD filtering.


c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:

Processing deviation function for model: Petrelli et al., 2020 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Petrelli et al., 2020 (cpx_only)_deviation_function_v_ind.pkl


...

Processing deviation function for model: Jorgenson et al., 2022 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Jorgenson et al., 2022 (cpx_only)_deviation_function_v_ind.pkl


R[write to console]: 
Attaching package: 'zoo'


R[write to console]: The following objects are masked from 'package:base':

    as.Date, as.Date.numeric


R[write to console]: 
Attaching package: 'PerformanceAnalytics'


R[write to console]: The following objects are masked from 'package:EnvStats':

    kurtosis, skewness


R[write to console]: The following object is masked from 'package:graphics':

    legend


R[write to console]: Loading required package: readxl



Processing deviation function for model: Ágreda-López et al., 2024 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Ágreda-López et al., 2024 (cpx_only)_deviation_function_v_ind.pkl
Processing deviation function for model: Wang et al., 2021 (cpx_only)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Wang et al., 2021 (cpx_only)_deviation_function_v_ind.pkl
Processing deviation function for model: Putirka, 2008 eq33_T; eq31_P (cpx_liq)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Putirka, 2008 eq33_T; eq31_P (cpx_liq)_deviation_function_v_ind.pkl
No OOD detector found for model: Putirka, 2008 eq33_T; eq31_P (cpx_liq), proceeding without OOD filtering.


c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\13493\anaconda3\envs\GPU\Lib\site-packages\pandas\core\arraylike.p

Processing deviation function for model: Petrelli et al., 2020 (cpx_liq)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Petrelli et al., 2020 (cpx_liq)_deviation_function_v_ind.pkl
Processing deviation function for model: Jorgenson et al., 2022 (cpx_liq)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Jorgenson et al., 2022 (cpx_liq)_deviation_function_v_ind.pkl
Processing deviation function for model: Ágreda-López et al., 2024 (cpx_liq)
c:\users\13493\documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\Deviation_functions\T_Ágreda-López et al., 2024 (cpx_liq)_deviation_function_v_ind.pkl
